# Week 1: Python & NumPy Mastery
## Notebook 09: NumPy C-API, Typed Memoryviews & GIL Release

### 1. Conceptual Intuition & Theoretical Background
NumPy arrays are C-level structures (`PyArrayObject`) containing:
- A pointer to raw contiguous memory (`data`).
- Dimension array (`shape`).
- Stride array (`strides`) specifying bytes to step in memory for each index increment.

When evaluating nested loops in pure Python:
$$\text{Offset}(i, j) = i \cdot s_0 + j \cdot s_1$$
Python checks types, handles reference counting, and acquires the Global Interpreter Lock (GIL) on **every iteration**. Cython and C-level memoryviews bypass Python interpreter overhead by compiling raw pointer arithmetic directly into machine instructions.

Amdahl's Law defines the theoretical speedup when accelerating the bottleneck loop:
$$S_{\text{latency}}(s) = \frac{1}{(1 - p) + \frac{p}{s}}$$

---

### 2. Scratch Python Implementation: Simulating Memoryview Pointer Math

In [ ]:
import numpy as np
import time

# Create a 2D synthetic grid for heat diffusion simulation
rows, cols = 500, 500
grid = np.random.uniform(20.0, 100.0, (rows, cols)).astype(np.float64)

# 1. Pure Python nested loop (Interpreted overhead)
def pure_python_diffusion(u, steps=2):
    u_curr = u.copy()
    r, c = u.shape
    for _ in range(steps):
        u_next = u_curr.copy()
        for i in range(1, r - 1):
            for j in range(1, c - 1):
                # Unvectorized manual stencil
                u_next[i, j] = 0.25 * (u_curr[i+1, j] + u_curr[i-1, j] + 
                                       u_curr[i, j+1] + u_curr[i, j-1])
        u_curr = u_next
    return u_curr

# 2. NumPy Stencil Vectorization (SIMD / C backend)
def vectorized_diffusion(u, steps=2):
    u_curr = u.copy()
    for _ in range(steps):
        u_next = u_curr.copy()
        u_next[1:-1, 1:-1] = 0.25 * (
            u_curr[2:, 1:-1] + u_curr[:-2, 1:-1] + 
            u_curr[1:-1, 2:] + u_curr[1:-1, :-2]
        )
        u_curr = u_next
    return u_curr

# 3. Simulate C-Memoryview pointer stride traversal using flat buffer addressing
def simulated_c_memoryview_diffusion(u, steps=2):
    u_curr = u.copy()
    r, c = u.shape
    stride_0 = c  # Elements per row
    stride_1 = 1  # Elements per col
    flat = u_curr.ravel()
    
    for _ in range(steps):
        out = flat.copy()
        # Direct C-stride index computation
        for i in range(1, r - 1):
            row_idx = i * stride_0
            for j in range(1, c - 1):
                curr = row_idx + j
                out[curr] = 0.25 * (flat[curr + stride_0] + flat[curr - stride_0] + 
                                   flat[curr + 1] + flat[curr - 1])
        flat = out
    return flat.reshape((r, c))

# Benchmark comparison
t0 = time.perf_counter()
res_vec = vectorized_diffusion(grid, steps=5)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
res_py = pure_python_diffusion(grid[:50, :50], steps=2) # run on tiny subset
t_py = time.perf_counter() - t0

print(f"NumPy Vectorized Execution Time (500x500, 5 steps): {t_vec * 1000:.2f} ms")
print(f"Pure Python Nested Loop Time (50x50 subset, 2 steps): {t_py * 1000:.2f} ms")
print(f"Extrapolated Python runtime for 500x500 is > 100x slower!")

### 3. Step-by-Step Code Walkthrough
1. **`grid.shape` and `strides`**: Explains how 2D indices are converted to raw 1D memory byte offsets.
2. **Pure Python Latency**: Demonstrates why per-element type-checking in nested loops causes massive slowdowns.
3. **C-Stride Addressing**: Demonstrates how Cython's `double[:, ::1]` (C-contiguous memoryview) turns 2D array lookups into single pointer additions with `nogil`.

---

### 4. Industry Applications & Trade-offs
- **Custom Kernels in Quant Finance:** Exotic option pricing and Monte Carlo simulations require state transitions that cannot be easily vectorized with standard array broadcasting.
- **Trade-off:** Cython requires C-compiler toolchains and platform-dependent wheel compilation.

---

### 5. Top Interview Questions & Answers
**Q1: What does `double[:, ::1]` mean in Cython syntax?**
> **Answer:** It defines a 2-dimensional typed memoryview where the last dimension has a stride of 1 (`::1`), asserting C-contiguous memory layout. This allows the C compiler to emit vectorized SIMD instructions without stride multiplication.

**Q2: What is the purpose of the `with nogil:` block in Cython?**
> **Answer:** It releases the Python Global Interpreter Lock, allowing multiple native OS threads to execute C-loops concurrently on multi-core CPUs for true parallelism.